# Handeling Outliers

Fill `path` with your dataframe file path, then run the notebook. The notebook is saved with no executed cells and no outputs.

This notebook prepares the dataframe before temporal split. It does not train a model, does not calculate course difficulty, and does not use a scaler for outliers.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)


In [ ]:
# Put your file path here.
path = r'D:\AI\Real projects\Academic_Advisor\data\audit\df_crg_add_acd.parquet'

# Main rules.
MAX_ALLOWED_SEMESTER_CREDITS = 25
MAX_ALLOWED_SEMESTER_COURSES = 8
FAIL_CREDITS_CAP = 120
HIGH_COURSE_CREDITS_VALUE = 24
MIN_VALID_PREV_GPA_POINTS = 0

# Plot sample size. This keeps plots readable if the data is very large.
PLOT_SAMPLE_SIZE = 10_000


In [ ]:
# Load the dataframe.
file_path = Path(path)

if not path:
    raise ValueError('Fill the path variable before running this notebook.')

if file_path.suffix.lower() in ['.parquet', '.pq']:
    df_feature = pd.read_parquet(file_path)
elif file_path.suffix.lower() == '.csv':
    df_feature = pd.read_csv(file_path)
elif file_path.suffix.lower() in ['.xlsx', '.xls']:
    df_feature = pd.read_excel(file_path)
elif file_path.suffix.lower() in ['.pkl', '.pickle']:
    df_feature = pd.read_pickle(file_path)
else:
    raise ValueError(f'Unsupported file type: {file_path.suffix}')

df_feature.head()


In [ ]:
# Make a working copy.
df = df_feature.copy()
original_columns = set(df.columns)
original_row_count = len(df)

# These columns must exist.
required_columns = [
    'student_id',
    'course_id',
    'degree_id',
    'part_id',
    'final_mark',
    'course_credits',
    'semester_reg_credits',
]

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

# These columns are useful, but if they do not exist we create simple default values.
optional_defaults = {
    'attempt_number': 1,
    'gpa_points': np.nan,
    'semester_pass_credits': np.nan,
    'prev_gpa_points': np.nan,
    'start_agpa_points': np.nan,
    'start_total_in_courses': 0,
    'start_total_in_credits': 0,
    'semester_reg_courses': np.nan,
    'total_pass_credits': np.nan,
    'total_fail_credits': 0,
    'reg_total_semesters': 0,
    'start_level_name_pl': pd.NA,
    'start_part_id': pd.NA,
    'requirement_type_id': pd.NA,
    'degree_requirement_credits_count': np.nan,
}

for col, value in optional_defaults.items():
    if col not in df.columns:
        df[col] = value

# Convert numeric columns to numbers.
numeric_columns = [
    'final_mark',
    'course_credits',
    'attempt_number',
    'gpa_points',
    'semester_reg_credits',
    'semester_pass_credits',
    'prev_gpa_points',
    'start_agpa_points',
    'start_total_in_courses',
    'start_total_in_credits',
    'semester_reg_courses',
    'total_pass_credits',
    'total_fail_credits',
    'reg_total_semesters',
    'requirement_type_id',
    'degree_requirement_credits_count',
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Original rows:', original_row_count)
print('Columns:', len(df.columns))
display(df.head())
display(df.isna().sum().sort_values(ascending=False).to_frame('null_count'))


In [ ]:
# Look at the start level values before mapping them.
start_level_values = (
    df['start_level_name_pl']
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .sort_values()
)

display(start_level_values.to_frame('start_level_name_pl'))

# Broad before-cleaning plots were removed. The only plots kept in this notebook
# focus on the interruption and previous-GPA cleaning changes below.


In [ ]:
# 1. Create policy and audit flags on the full dataframe.
# course_credits = 24 is valid. Keep it and expose it as a feature/audit flag.
df['is_high_credit_course'] = (df['course_credits'] == HIGH_COURSE_CREDITS_VALUE).astype(int)

df['over_policy_semester_credits'] = df['semester_reg_credits'] > MAX_ALLOWED_SEMESTER_CREDITS

if 'semester_reg_courses' in df.columns:
    df['over_policy_semester_courses'] = df['semester_reg_courses'] > MAX_ALLOWED_SEMESTER_COURSES
else:
    print('WARNING: semester_reg_courses is missing; course-count overload check was skipped.')
    df['over_policy_semester_courses'] = False

df['exclude_over_policy_semester'] = df['over_policy_semester_credits'] | df['over_policy_semester_courses']

df['total_fail_credits'] = df['total_fail_credits'].fillna(0)
df['is_extreme_fail_history'] = (df['total_fail_credits'] > FAIL_CREDITS_CAP).astype(int)
df['total_fail_credits_capped'] = df['total_fail_credits'].clip(upper=FAIL_CREDITS_CAP)

# Freeze the full audit dataframe before Model A feature engineering.
df_model_audit = df.copy()
df_excluded_over_policy = df_model_audit[df_model_audit['exclude_over_policy_semester']].copy()
df_primary = df_model_audit[~df_model_audit['exclude_over_policy_semester']].copy()

print('Full audit rows:', len(df_model_audit))
print('Excluded over-policy rows:', len(df_excluded_over_policy))
print('Primary Model A rows:', len(df_primary))

# 2. Build student-semester features from df_primary only.
semester_key = ['student_id', 'degree_id', 'part_id']
student_degree_key = ['student_id', 'degree_id']
semester_value_columns = [
    'gpa_points',
    'semester_reg_credits',
    'semester_pass_credits',
    'start_total_in_courses',
    'start_total_in_credits',
    'start_agpa_points',
    'prev_gpa_points',
]
semester_value_columns = [col for col in semester_value_columns if col in df_primary.columns]

# Audit duplicated course rows before aggregation. These columns should be repeated
# within each student-degree-semester, so conflicting values are warnings.
conflict_frames = []
for col in semester_value_columns:
    col_conflicts = (
        df_primary.groupby(semester_key, dropna=False)[col]
        .nunique(dropna=False)
        .reset_index(name='unique_value_count')
    )
    col_conflicts = col_conflicts[col_conflicts['unique_value_count'] > 1].copy()
    if not col_conflicts.empty:
        col_conflicts.insert(0, 'column', col)
        conflict_frames.append(col_conflicts)

if conflict_frames:
    semester_conflict_report = pd.concat(conflict_frames, ignore_index=True)
    print('WARNING: semester-level columns have conflicting values within some student-degree-semesters. Aggregation will use max.')
    display(semester_conflict_report.head(25))
else:
    semester_conflict_report = pd.DataFrame(columns=['column'] + semester_key + ['unique_value_count'])
    print('No semester-level conflicts detected before aggregation.')

# Use max consistently because these semester attributes should be repeated per course row.
semester_df = (
    df_primary.groupby(semester_key, dropna=False, as_index=False)
    .agg({col: 'max' for col in semester_value_columns})
)

semester_df['_student_sort'] = semester_df['student_id'].astype('string')
semester_df['_degree_sort'] = semester_df['degree_id'].astype('string')
semester_df['_part_sort_numeric'] = pd.to_numeric(semester_df['part_id'], errors='coerce')
semester_df['_part_sort_text'] = semester_df['part_id'].astype('string')
semester_df = (
    semester_df
    .sort_values(
        ['_student_sort', '_degree_sort', '_part_sort_numeric', '_part_sort_text'],
        kind='mergesort',
        na_position='last',
    )
    .drop(columns=['_student_sort', '_degree_sort', '_part_sort_numeric', '_part_sort_text'])
    .reset_index(drop=True)
)

semester_df['is_interruption_semester'] = (
    semester_df['semester_reg_credits'].fillna(0).gt(0)
    & semester_df['semester_pass_credits'].fillna(0).eq(0)
    & semester_df['gpa_points'].eq(0).fillna(False)
).astype(int)

semester_group = semester_df.groupby(student_degree_key, dropna=False, sort=False)
semester_df['prev_semester_was_interruption'] = (
    semester_group['is_interruption_semester']
    .shift(1)
    .fillna(0)
    .astype(int)
)
semester_df['prior_interruption_count'] = (
    semester_group['is_interruption_semester'].cumsum() - semester_df['is_interruption_semester']
).astype(int)


def _prior_consecutive_interruptions(flags):
    counts = []
    current_run = 0
    for flag in flags.fillna(0).astype(int):
        counts.append(current_run)
        current_run = current_run + 1 if flag == 1 else 0
    return pd.Series(counts, index=flags.index)


semester_df['consecutive_interruption_count'] = (
    semester_group['is_interruption_semester']
    .transform(_prior_consecutive_interruptions)
    .astype(int)
)

semester_df['no_previous_progress'] = (
    semester_df['start_total_in_courses'].fillna(0).eq(0)
    & semester_df['start_total_in_credits'].fillna(0).eq(0)
).astype(int)
semester_df['is_first_active_semester'] = semester_df['no_previous_progress']

# Ignore interruption semesters, then shift the filled GPA to get the nearest valid previous GPA.
semester_df['semester_gpa_for_history'] = semester_df['gpa_points'].where(
    semester_df['is_interruption_semester'].eq(0)
)
semester_group = semester_df.groupby(student_degree_key, dropna=False, sort=False)
semester_df['_filled_valid_gpa'] = semester_group['semester_gpa_for_history'].ffill()
semester_group = semester_df.groupby(student_degree_key, dropna=False, sort=False)
semester_df['last_valid_gpa_before_current_semester'] = semester_group['_filled_valid_gpa'].shift(1)
semester_df = semester_df.drop(columns=['_filled_valid_gpa'])

# 3. Merge semester-level features back to df_primary with safe string keys.
merge_key_columns = ['_student_merge_key', '_degree_merge_key', '_part_merge_key']
source_key_columns = ['student_id', 'degree_id', 'part_id']
for merge_col, source_col in zip(merge_key_columns, source_key_columns):
    df_primary[merge_col] = df_primary[source_col].astype('string').fillna('__MISSING__')
    semester_df[merge_col] = semester_df[source_col].astype('string').fillna('__MISSING__')

semester_feature_columns = [
    'is_interruption_semester',
    'prev_semester_was_interruption',
    'prior_interruption_count',
    'consecutive_interruption_count',
    'no_previous_progress',
    'is_first_active_semester',
    'last_valid_gpa_before_current_semester',
]
semester_feature_columns = [col for col in semester_feature_columns if col in semester_df.columns]
semester_feature_df = semester_df[merge_key_columns + semester_feature_columns].drop_duplicates(merge_key_columns).copy()

merge_preview = df_primary[merge_key_columns].merge(
    semester_feature_df[merge_key_columns],
    on=merge_key_columns,
    how='left',
    indicator=True,
)
semester_feature_merge_counts = merge_preview['_merge'].value_counts()
print('Semester feature merge check:')
display(semester_feature_merge_counts.to_frame('row_count'))

left_only_keys = merge_preview.loc[merge_preview['_merge'].eq('left_only'), merge_key_columns].drop_duplicates()
if not left_only_keys.empty:
    print('WARNING: Some df_primary rows did not match semester_df. Example keys:')
    display(left_only_keys.head(25))

df_primary = df_primary.merge(
    semester_feature_df,
    on=merge_key_columns,
    how='left',
    validate='many_to_one',
    indicator='_semester_feature_merge',
)
print('Actual semester feature merge result:')
display(df_primary['_semester_feature_merge'].value_counts().to_frame('row_count'))

semester_last_valid_non_null = int(semester_df['last_valid_gpa_before_current_semester'].notna().sum())
primary_last_valid_non_null = int(df_primary['last_valid_gpa_before_current_semester'].notna().sum())
print('Non-null last_valid_gpa_before_current_semester in semester_df:', semester_last_valid_non_null)
print('Non-null last_valid_gpa_before_current_semester in df_primary:', primary_last_valid_non_null)

if semester_last_valid_non_null > 0 and primary_last_valid_non_null == 0:
    raise ValueError('last_valid_gpa_before_current_semester merge failed: semester_df has values but df_primary has none.')

for col in [c for c in semester_feature_columns if c != 'last_valid_gpa_before_current_semester']:
    df_primary[col] = df_primary[col].fillna(0).astype(int)

df_primary = df_primary.drop(columns=merge_key_columns + ['_semester_feature_merge'], errors='ignore')

# 4. Clean prev_gpa_points on df_primary only. Keep raw prev_gpa_points unchanged.
# Use only positive GPA values; zero is treated as an old-system/system placeholder.
# Source order: raw prev_gpa_points -> nearest previous valid GPA -> start_agpa_points.
# 4. Clean prev_gpa_points on df_primary only. Keep raw prev_gpa_points unchanged.
# Use only positive GPA values; zero is treated as an old-system/system placeholder.
# Source order:
# raw prev_gpa_points > 0
# -> last_valid_gpa_before_current_semester > 0
# -> start_agpa_points > 0
# -> 0 final fallback

MIN_VALID_PREV_GPA_POINTS = 0

raw_prev_gpa = pd.to_numeric(df_primary["prev_gpa_points"], errors="coerce")
last_valid_prev_gpa = pd.to_numeric(df_primary["last_valid_gpa_before_current_semester"], errors="coerce")
start_agpa_points = pd.to_numeric(df_primary["start_agpa_points"], errors="coerce")

df_primary["prev_gpa_points_missing"] = raw_prev_gpa.isna().astype(int)
df_primary["prev_gpa_points_zero"] = raw_prev_gpa.eq(0).fillna(False).astype(int)

df_primary["prev_gpa_invalid_zero_case"] = (
    raw_prev_gpa.eq(0).fillna(False)
    & df_primary["is_first_active_semester"].ne(1)
).astype(int)

df_primary["prev_gpa_points_clean"] = np.nan
df_primary["prev_gpa_fill_source"] = pd.Series(pd.NA, index=df_primary.index, dtype="string")

prev_gpa_sources = [
    ("raw_prev_gpa", raw_prev_gpa),
    ("last_valid_gpa_before_current_semester", last_valid_prev_gpa),
    ("start_agpa_points", start_agpa_points),
]

for source_name, source_values in prev_gpa_sources:
    needs_value = df_primary["prev_gpa_points_clean"].isna()
    source_has_value = source_values.gt(MIN_VALID_PREV_GPA_POINTS).fillna(False)

    use_source = needs_value & source_has_value

    df_primary.loc[use_source, "prev_gpa_points_clean"] = source_values.loc[use_source]
    df_primary.loc[use_source, "prev_gpa_fill_source"] = source_name

# Final fallback: if still NaN, set to 0
needs_zero_fallback = df_primary["prev_gpa_points_clean"].isna()

df_primary.loc[needs_zero_fallback, "prev_gpa_points_clean"] = 0
df_primary.loc[needs_zero_fallback, "prev_gpa_fill_source"] = "zero_fallback"

# Audit flags
df_primary["prev_gpa_replaced_due_to_invalid_zero"] = (
    df_primary["prev_gpa_invalid_zero_case"].eq(1)
    & df_primary["prev_gpa_fill_source"].ne("raw_prev_gpa")
).astype(int)

# Since support confirmed prev_gpa_points == 0 is an old/system case,
# do not treat zero as real academic performance.
df_primary["prev_gpa_actual_zero_performance"] = 0

# Final validation
remaining_nan_count = df_primary["prev_gpa_points_clean"].isna().sum()
print("Remaining NaN in prev_gpa_points_clean:", remaining_nan_count)

assert remaining_nan_count == 0, "prev_gpa_points_clean still contains NaN values."

print("prev_gpa_fill_source distribution:")
display(df_primary["prev_gpa_fill_source"].value_counts(dropna=False).to_frame("row_count"))

print("prev_gpa_points_clean summary:")
display(df_primary["prev_gpa_points_clean"].describe())

display(
    df_primary.loc[
        df_primary["prev_gpa_fill_source"].eq("zero_fallback"),
        [
            "student_id",
            "degree_id",
            "part_id",
            "prev_gpa_points",
            "last_valid_gpa_before_current_semester",
            "start_agpa_points",
            "prev_gpa_points_clean",
            "prev_gpa_fill_source",
            "is_first_active_semester",
        ]
    ].head(25)
)
# 5. Split part_id and start_part_id into year and semester columns. Keep raw part_id.
part_text = df_primary['part_id'].astype('string').str.extract(r'(\d{4})\D*(\d+)')
df_primary['part_year'] = pd.to_numeric(part_text[0], errors='coerce')
df_primary['part_semester'] = pd.to_numeric(part_text[1], errors='coerce')

start_part_text = df_primary['start_part_id'].astype('string').str.extract(r'(\d{4})\D*(\d+)')
df_primary['start_year'] = pd.to_numeric(start_part_text[0], errors='coerce')
df_primary['start_semester'] = pd.to_numeric(start_part_text[1], errors='coerce')

# 6. Convert start_level_name_pl to academic order. Missing or unknown levels become 0.
# Values are lower case because start_level_clean is stripped and lowercased below.
level_order = {
    "first year": 1,
    "second year": 2,
    "third year": 3,
    "fourth year": 4,
    "fifth year": 5,
    "sixth year": 6,
}

start_level_clean = (
    df_primary["start_level_name_pl"]
    .astype("string")
    .str.strip()
    .str.lower()
)

start_level_clean = start_level_clean.mask(start_level_clean.fillna("").eq(""))

df_primary["start_level_missing"] = start_level_clean.isna().astype(int)

df_primary["start_level_ord"] = (
    start_level_clean
    .map(level_order)
    .fillna(0)
    .astype(int)
)

# 7. Requirement features.
df_primary['requirement_type_missing'] = df_primary['requirement_type_id'].isna().astype(int)
df_primary['requirement_type_id'] = df_primary['requirement_type_id'].fillna(-1).astype('Int64')

df_primary['degree_requirement_credits_count_missing'] = df_primary['degree_requirement_credits_count'].isna().astype(int)
df_primary['degree_requirement_credits_count'] = df_primary['degree_requirement_credits_count'].fillna(0)

df_primary['course_share_of_requirement'] = 0.0
valid_requirement = df_primary['degree_requirement_credits_count'] > 0
df_primary.loc[valid_requirement, 'course_share_of_requirement'] = (
    df_primary.loc[valid_requirement, 'course_credits'] / df_primary.loc[valid_requirement, 'degree_requirement_credits_count']
)

df_primary['requirement_size_bucket'] = pd.cut(
    df_primary['degree_requirement_credits_count'],
    bins=[-0.01, 0, 12, 24, 60, np.inf],
    labels=['none_or_unknown', 'small', 'medium', 'large', 'very_large'],
).astype('string').fillna('unknown')

# 8. Keep this key for later encoding or course statistics. Do not calculate course difficulty here.
df_primary['degree_course_key'] = df_primary['degree_id'].astype('string') + '__' + df_primary['course_id'].astype('string')

# 9. Row-wise failure ratio.
fail_credit_denominator = df_primary['start_total_in_credits'].fillna(0) + df_primary['total_fail_credits_capped'].fillna(0)
df_primary['fail_credit_ratio_capped'] = 0.0
valid_fail_ratio = fail_credit_denominator > 0
df_primary.loc[valid_fail_ratio, 'fail_credit_ratio_capped'] = (
    df_primary.loc[valid_fail_ratio, 'total_fail_credits_capped'] / fail_credit_denominator.loc[valid_fail_ratio]
)

display(pd.Series(level_order, name='start_level_ord').rename_axis('start_level_name_pl').reset_index())
display(df_primary.head())


In [ ]:
df.head()

In [ ]:
# Audit reports.
high_credit_course_report = (
    df_model_audit[df_model_audit['is_high_credit_course'] == 1]
    .groupby(['course_id', 'course_credits'], dropna=False)
    .size()
    .reset_index(name='row_count')
    .sort_values('row_count', ascending=False)
)

interruption_semester_columns = [
    'student_id',
    'degree_id',
    'part_id',
    'gpa_points',
    'semester_reg_credits',
    'semester_pass_credits',
    'is_interruption_semester',
]
interruption_semester_report = semester_df.loc[
    semester_df['is_interruption_semester'] == 1,
    [col for col in interruption_semester_columns if col in semester_df.columns],
].copy()

prev_gpa_columns = [
    'student_id',
    'course_id',
    'degree_id',
    'part_id',
    'prev_gpa_points',
    'prev_gpa_points_clean',
    'start_agpa_points',
    'last_valid_gpa_before_current_semester',
    'prev_semester_was_interruption',
    'prior_interruption_count',
    'consecutive_interruption_count',
    'start_total_in_courses',
    'start_total_in_credits',
    'no_previous_progress',
    'is_first_active_semester',
    'prev_gpa_fill_source',
]
prev_gpa_columns = [col for col in prev_gpa_columns if col in df_primary.columns]
changed_prev_gpa_mask = (
    df_primary['prev_gpa_points'].isna()
    | df_primary['prev_gpa_points'].ne(df_primary['prev_gpa_points_clean']).fillna(False)
)
prev_gpa_cleaning_report = df_primary.loc[changed_prev_gpa_mask, prev_gpa_columns].copy()

prev_gpa_source_summary = (
    df_primary['prev_gpa_fill_source']
    .value_counts(dropna=False)
    .rename_axis('prev_gpa_fill_source')
    .reset_index(name='row_count')
)

# Backward-compatible alias for older notebook references.
prev_gpa_zero_report = prev_gpa_cleaning_report

cleaning_summary = pd.DataFrame({
    'metric': [
        'full_row_count',
        'primary_model_row_count',
        'excluded_over_policy_row_count',
        'high_credit_course_row_count',
        'interruption_semester_count',
        'unique_students_with_interruption_semesters',
        'rows_prev_gpa_changed_by_fallback',
        'rows_prev_gpa_missing_after_fallbacks',
        'rows_prev_gpa_clean_zero_values',
        'last_valid_gpa_before_current_semester_non_null_rows',
    ],
    'value': [
        len(df_model_audit),
        len(df_primary),
        len(df_excluded_over_policy),
        int(df_model_audit['is_high_credit_course'].sum()),
        int(semester_df['is_interruption_semester'].sum()),
        int(semester_df.loc[semester_df['is_interruption_semester'] == 1, 'student_id'].nunique(dropna=True)),
        int(changed_prev_gpa_mask.sum()),
        int(df_primary['prev_gpa_fill_source'].eq('missing_after_fallbacks').sum()),
        int(df_primary['prev_gpa_points_clean'].eq(0).fillna(False).sum()),
        int(df_primary['last_valid_gpa_before_current_semester'].notna().sum()),
    ],
})

display(cleaning_summary)
display(prev_gpa_source_summary)
display(df_primary['prev_gpa_points_clean'].describe().to_frame('prev_gpa_points_clean'))
display(df_excluded_over_policy.head(25))
display(interruption_semester_report.head(25))
display(prev_gpa_cleaning_report.head(25))

if not semester_conflict_report.empty:
    display(semester_conflict_report.head(25))
else:
    print('No semester conflicts to display.')

display(high_credit_course_report.head(25))


In [ ]:
# Final Model A dataframes. df_model_audit remains full; Model A uses df_primary only.
df_model_a = df_primary.copy()

model_a_features = [
    'course_id',
    'degree_id',
    'part_id',
    'part_year',
    'part_semester',
    'course_credits',
    'is_high_credit_course',
    'attempt_number',
    'is_interruption_semester',
    'prev_semester_was_interruption',
    'prior_interruption_count',
    'consecutive_interruption_count',
    'no_previous_progress',
    'is_first_active_semester',
    'prev_gpa_points_clean',
    'start_agpa_points',
    'start_total_in_courses',
    'start_total_in_credits',
    'semester_reg_courses',
    'semester_reg_credits',
    'total_fail_credits_capped',
    'is_extreme_fail_history',
    'fail_credit_ratio_capped',
    'reg_total_semesters',
    'start_level_ord',
    'start_level_missing',
    'requirement_type_id',
    'requirement_type_missing',
    'degree_requirement_credits_count',
    'degree_requirement_credits_count_missing',
    'course_share_of_requirement',
    'requirement_size_bucket',
]

model_a_features = [col for col in model_a_features if col in df_model_a.columns]

y_model_a = df_model_a['final_mark'].copy()
X_model_a = df_model_a[model_a_features].copy()

# Make sure blocked raw/leakage columns are not in X_model_a.
X_model_a = X_model_a.drop(
    columns=[
        'final_mark',
        'prev_gpa_points',
        'prev_gpa_fill_source',
        'last_valid_gpa_before_current_semester',
        'degree_course_key',
        'total_pass_credits',
        'total_fail_credits',
        'start_level_name_pl',
    ],
    errors='ignore',
)

assert len(X_model_a) == len(y_model_a)
assert 'final_mark' not in X_model_a.columns
assert 'prev_gpa_points' not in X_model_a.columns
assert 'prev_gpa_fill_source' not in X_model_a.columns
assert 'last_valid_gpa_before_current_semester' not in X_model_a.columns
assert 'degree_course_key' not in X_model_a.columns
assert 'total_fail_credits' not in X_model_a.columns
# assert not df_model_a['prev_gpa_points_clean'].eq(0).fillna(False).any()

print('df_model_audit rows:', len(df_model_audit))
print('df_excluded_over_policy rows:', len(df_excluded_over_policy))
print('df_model_a rows:', len(df_model_a))
print('X_model_a shape:', X_model_a.shape)
print('y_model_a shape:', y_model_a.shape)

display(X_model_a.head())
display(y_model_a.head().to_frame('final_mark'))


In [ ]:
X_model_a.columns

In [ ]:
# Final summaries.
final_null_summary = (
    df_model_a[X_model_a.columns.tolist() + ['final_mark']]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame('null_count')
)

final_feature_list = pd.Series(X_model_a.columns, name='feature').to_frame()

display(cleaning_summary)
display(final_null_summary)
display(final_feature_list)


In [ ]:
# Important change plots only: interruption semesters and previous-GPA fallback cleaning on df_primary.
important_flag_columns = [
    'is_interruption_semester',
    'prev_semester_was_interruption',
    'is_first_active_semester',
]
important_flag_columns = [col for col in important_flag_columns if col in df_model_a.columns]

important_flag_counts = pd.DataFrame({
    'flag': important_flag_columns,
    'row_count': [int(df_model_a[col].sum()) for col in important_flag_columns],
})

if not important_flag_counts.empty:
    plt.figure(figsize=(11, 5))
    sns.barplot(
        data=important_flag_counts.sort_values('row_count', ascending=False),
        x='row_count',
        y='flag',
        color='#4C78A8',
    )
    plt.title('Important interruption and first-semester flags')
    plt.tight_layout()
    plt.show()

raw_prev_gpa = df_model_a['prev_gpa_points']
clean_prev_gpa = df_model_a['prev_gpa_points_clean']
changed_prev_gpa_mask = (
    raw_prev_gpa.isna()
    | raw_prev_gpa.ne(clean_prev_gpa).fillna(False)
)
changed_prev_gpa_plot = df_model_a.loc[changed_prev_gpa_mask].copy()

if changed_prev_gpa_plot.empty:
    print('No previous-GPA rows changed by the fallback rules.')
else:
    changed_prev_gpa_plot = changed_prev_gpa_plot.sample(
        n=min(len(changed_prev_gpa_plot), PLOT_SAMPLE_SIZE),
        random_state=42,
    )
    value_columns = [
        'prev_gpa_points',
        'prev_gpa_points_clean',
        'last_valid_gpa_before_current_semester',
    ]
    value_columns = [col for col in value_columns if col in changed_prev_gpa_plot.columns]
    gpa_plot_data = changed_prev_gpa_plot[value_columns].rename(columns={
        'prev_gpa_points': 'raw_prev_gpa_points',
        'prev_gpa_points_clean': 'clean_prev_gpa_points',
        'last_valid_gpa_before_current_semester': 'last_valid_gpa_before_current',
    }).melt(var_name='value_type', value_name='gpa_points')
    gpa_plot_data = gpa_plot_data.dropna(subset=['gpa_points'])

    if not gpa_plot_data.empty:
        plt.figure(figsize=(11, 5))
        sns.histplot(data=gpa_plot_data, x='gpa_points', hue='value_type', element='step', bins=30)
        plt.title('Previous-GPA values on rows changed by cleaning')
        plt.tight_layout()
        plt.show()

interruption_by_part = (
    semester_df.loc[semester_df['is_interruption_semester'] == 1]
    .groupby('part_id', dropna=False)
    .size()
    .reset_index(name='interruption_semester_count')
    .sort_values('interruption_semester_count', ascending=False)
    .head(25)
)

if interruption_by_part.empty:
    print('No interruption semesters detected in df_primary.')
else:
    interruption_by_part['part_id'] = interruption_by_part['part_id'].astype('string').fillna('missing')
    plt.figure(figsize=(11, 5))
    sns.barplot(data=interruption_by_part, x='interruption_semester_count', y='part_id', color='#F58518')
    plt.title('Top primary semesters with interruption records')
    plt.tight_layout()
    plt.show()


In [ ]:
df_model_a.to_parquet(r'D:\AI\Real projects\Academic_Advisor\data\final\without_)outliers.parquet')